# Pokémon ETL — Extraction Demo

This notebook demonstrates how data is extracted from PokeAPI and stored as raw Parquet data.

## Extraction Flow

PokeAPI → Extract → Raw Parquet

In [55]:
import sys
from pathlib import Path
import pandas as pd
# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from extract import (
    create_session,
    get_pokemon,
    extract_pokemon,
    extract_species,
    get_move,
    extract_moves
)

from logger import setup_logger

## 1. Create API Session

A reusable session handles common request settings such as retries, timeouts and headers.

In [56]:
session = create_session()
logger = setup_logger()

print("API session created successfully.")

API session created successfully.


## 2. Discover Pokémon URLs

Pagination is used to collect the resource URLs before extracting the actual records.

In [57]:
pokemon_urls, failed_pages = get_pokemon(
    session,
    logger
)

print("Pokemon URLs:", len(pokemon_urls))
print("Failed pages:", len(failed_pages))

Pokemon URLs: 1351
Failed pages: 0


## 3. Extract Pokémon Data

In [58]:
# using a small sample for the live demonstration
demo_urls = pokemon_urls[:5]

pokemon_data, failed_pokemon = extract_pokemon(
    session,
    demo_urls,
    logger
)

pokemon_df = pd.DataFrame(pokemon_data)

print("Records extracted:", len(pokemon_data))
print("Failed records:", len(failed_pokemon))

pokemon_df[
    ["id", "name", "height", "weight", "base_experience"]
]

2026-10-03 14:08:00 | INFO | pokemon extraction completed | success=1 failed=0
INFO:pokemon_pipeline:pokemon extraction completed | success=1 failed=0
2026-10-03 14:08:00 | INFO | pokemon extraction completed | success=2 failed=0
INFO:pokemon_pipeline:pokemon extraction completed | success=2 failed=0
2026-10-03 14:08:00 | INFO | pokemon extraction completed | success=3 failed=0
INFO:pokemon_pipeline:pokemon extraction completed | success=3 failed=0
2026-10-03 14:08:01 | INFO | pokemon extraction completed | success=4 failed=0
INFO:pokemon_pipeline:pokemon extraction completed | success=4 failed=0
2026-10-03 14:08:01 | INFO | pokemon extraction completed | success=5 failed=0
INFO:pokemon_pipeline:pokemon extraction completed | success=5 failed=0


Records extracted: 5
Failed records: 0


,id,name,height,weight,base_experience
0,1,bulbasaur,7,69,64
1,2,ivysaur,10,130,142
2,3,venusaur,20,1000,236
3,4,charmander,6,85,62
4,5,charmeleon,11,190,142


 ## THREAD POOL EXECUTOR

The discovered URLs are passed to `extract_pokemon()`.

The extraction function uses bounded concurrency with `ThreadPoolExecutor`.

Instead of processing:

Request 1 → wait → Request 2 → wait → Request 3

multiple requests can be processed concurrently using 10 workers.

This is useful because API extraction is mainly network I/O bound.                  
                   
                    Pokémon URLs
                         │
                         ▼
              ThreadPoolExecutor
                    10 workers
                         │
          ┌──────────────┼──────────────┐
          ▼              ▼              ▼
       Worker 1       Worker 2       Worker 3 ... Worker 10
          │              │              │
          └──────────────┼──────────────┘
                         ▼
                  Pokémon JSON data

## 4. Discover Related Resources

Pokémon records contain references to their species and moves.

In [59]:
species_urls = list({
    pokemon["species"]["url"]
    for pokemon in pokemon_data
    if pokemon.get("species")
    and pokemon["species"].get("url")
})

move_urls = get_move(pokemon_data)

print("Species URLs:", len(species_urls))
print("Move URLs:", len(move_urls))

Species URLs: 5
Move URLs: 159


In [60]:
species_data, failed_species = extract_species(
    session,
    species_urls,
    logger
)

move_data, failed_moves = extract_moves(
    session,
    move_urls,
    logger
)

print("Species extracted:", len(species_data))
print("Moves extracted:", len(move_data))

2026-10-03 14:08:01 | INFO | pokemon species extraction completed | success=1 | failed=0 
INFO:pokemon_pipeline:pokemon species extraction completed | success=1 | failed=0 
2026-10-03 14:08:01 | INFO | pokemon species extraction completed | success=2 | failed=0 
INFO:pokemon_pipeline:pokemon species extraction completed | success=2 | failed=0 
2026-10-03 14:08:01 | INFO | pokemon species extraction completed | success=3 | failed=0 
INFO:pokemon_pipeline:pokemon species extraction completed | success=3 | failed=0 
2026-10-03 14:08:01 | INFO | pokemon species extraction completed | success=4 | failed=0 
INFO:pokemon_pipeline:pokemon species extraction completed | success=4 | failed=0 
2026-10-03 14:08:02 | INFO | pokemon species extraction completed | success=5 | failed=0 
INFO:pokemon_pipeline:pokemon species extraction completed | success=5 | failed=0 
2026-10-03 14:08:02 | INFO | pokemon move successfully extracted | success=1 | failed =0
INFO:pokemon_pipeline:pokemon move successfull

Species extracted: 5
Moves extracted: 159


In [61]:
print("First Pokémon:")
print(pokemon_data[0]["name"])

print("\nFirst Species:")
print(species_data[0]["name"])

print("\nFirst Move:")
print(move_data[0]["name"])

First Pokémon:
bulbasaur

First Species:
charmander

First Move:
scratch


In [62]:
pokemon_df = pd.DataFrame(pokemon_data)

pokemon_df[
    ["id", "name", "height", "weight", "base_experience"]
].head()

,id,name,height,weight,base_experience
0,1,bulbasaur,7,69,64
1,2,ivysaur,10,130,142
2,3,venusaur,20,1000,236
3,4,charmander,6,85,62
4,5,charmeleon,11,190,142


## 5. Raw Data

The complete extraction is stored in Parquet files and used as the input for the transformation stage. Showing the raw data by reading from parquet files .

In [63]:
RAW_DIR = PROJECT_ROOT / "raw"

pokemon_raw = pd.read_parquet(RAW_DIR / "pokemon.parquet")
species_raw = pd.read_parquet(RAW_DIR / "species.parquet")
moves_raw = pd.read_parquet(RAW_DIR / "moves.parquet")

print("Pokemon:", len(pokemon_raw))
print("Species:", len(species_raw))
print("Moves:", len(moves_raw))

Pokemon: 1351
Species: 1025
Moves: 833


This is the raw data i stored.

In [64]:
pokemon_raw.head(3)

,id,name,base_experience,height,is_default,order,weight,abilities,past_abilities,forms,...,held_items,location_area_encounters,moves,species,sprites,cries,stats,past_stats,types,past_types
0,1,bulbasaur,64.0,7,True,1,69,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'bulbasaur', 'url': 'https://pokeapi...",...,[],https://pokeapi.co/api/v2/pokemon/1/encounters,"[{'move': {'name': 'razor-wind', 'url': 'https...","{'name': 'bulbasaur', 'url': 'https://pokeapi....",{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 45, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'grass', 'url': ...",[]
1,2,ivysaur,142.0,10,True,2,130,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'ivysaur', 'url': 'https://pokeapi.c...",...,[],https://pokeapi.co/api/v2/pokemon/2/encounters,"[{'move': {'name': 'swords-dance', 'url': 'htt...","{'name': 'ivysaur', 'url': 'https://pokeapi.co...",{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 60, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'grass', 'url': ...",[]
2,3,venusaur,236.0,20,True,3,1000,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'venusaur', 'url': 'https://pokeapi....",...,[],https://pokeapi.co/api/v2/pokemon/3/encounters,"[{'move': {'name': 'swords-dance', 'url': 'htt...","{'name': 'venusaur', 'url': 'https://pokeapi.c...",{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 80, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'grass', 'url': ...",[]


## Extraction Complete

The raw API data is now available in Parquet and ready for transformation.

## IDEMPOTENCY & INCREMENTAL UPSERT LOGIC

In [65]:
# Primary Key used for upsert tracking
PRIMARY_KEY = "id"

# Storage list representing our target table (initially empty)
existing_data = []

In [66]:
batch_1 = [
    {"id": 1, "name": "bulbasaur", "height": 7, "weight": 69},
    {"id": 2, "name": "ivysaur", "height": 10, "weight": 130},
    {"id": 3, "name": "venusaur", "height": 20, "weight": 1000},
]

In [67]:
# Convert existing records to primary key map: {id: record_dict}
existing_records = {
    record.get(PRIMARY_KEY): record
    for record in existing_data
    if record.get(PRIMARY_KEY) is not None
}

In [68]:
inserted = 0
updated = 0
unchanged = 0

In [69]:
# Loop through incoming records and perform upsert logic
for record in batch_1:
    record_id = record.get(PRIMARY_KEY)
    if record_id is None:
        continue

    if record_id not in existing_records:
        existing_records[record_id] = record
        inserted += 1
    elif existing_records[record_id] != record:
        existing_records[record_id] = record
        updated += 1
    else:
        unchanged += 1

In [70]:
# Update master dataset
existing_data = list(existing_records.values())

In [71]:
print(f"Summary -> Inserted: {inserted} | Updated: {updated} | Unchanged: {unchanged}")
print(f"Total Dataset Count: {len(existing_data)}")

Summary -> Inserted: 3 | Updated: 0 | Unchanged: 0
Total Dataset Count: 3


## Re-running the code , to understand idempotency & incremental load

observe , we got unchanged 3 , so we are not rewriting the existing data . 

In [72]:
existing_records = {
    record.get(PRIMARY_KEY): record
    for record in existing_data
    if record.get(PRIMARY_KEY) is not None
}

inserted = 0
updated = 0
unchanged = 0

for record in batch_1:
    record_id = record.get(PRIMARY_KEY)
    if record_id is None:
        continue

    if record_id not in existing_records:
        existing_records[record_id] = record
        inserted += 1
    elif existing_records[record_id] != record:
        existing_records[record_id] = record
        updated += 1
    else:
        unchanged += 1

existing_data = list(existing_records.values())

print(f"Summary -> Inserted: {inserted} | Updated: {updated} | Unchanged: {unchanged}")
print(f"Total Dataset Count: {len(existing_data)}")

Summary -> Inserted: 0 | Updated: 0 | Unchanged: 3
Total Dataset Count: 3


## Re-running the code  , to understand idempotency and incremental loading

Here we update one existing row , and inserted one new row and old data is present and unchanged the data which is as it is same

In [73]:
batch_2 = [
    {"id": 1, "name": "bulbasaur", "height": 7, "weight": 69},
    {"id": 2, "name": "ivysaur", "height": 10, "weight": 135},
    {"id": 4, "name": "charmander", "height": 6, "weight": 85},
]

In [74]:
existing_records = {
    record.get(PRIMARY_KEY): record
    for record in existing_data
    if record.get(PRIMARY_KEY) is not None
}

inserted = 0
updated = 0
unchanged = 0

for record in batch_2:
    record_id = record.get(PRIMARY_KEY)
    if record_id is None:
        continue

    if record_id not in existing_records:
        existing_records[record_id] = record
        inserted += 1
    elif existing_records[record_id] != record:
        existing_records[record_id] = record
        updated += 1
    else:
        unchanged += 1

existing_data = list(existing_records.values())

print(f"Summary -> Inserted: {inserted} | Updated: {updated} | Unchanged: {unchanged}")
print(f"Total Dataset Count: {len(existing_data)}")

Summary -> Inserted: 1 | Updated: 1 | Unchanged: 1
Total Dataset Count: 4


final output after idempotancy & incremental loading

In [75]:
for row in existing_data:
    print(row)

{'id': 1, 'name': 'bulbasaur', 'height': 7, 'weight': 69}
{'id': 2, 'name': 'ivysaur', 'height': 10, 'weight': 135}
{'id': 3, 'name': 'venusaur', 'height': 20, 'weight': 1000}
{'id': 4, 'name': 'charmander', 'height': 6, 'weight': 85}
